## ReLoG - ID ablation

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import copy
import random
import math
import os

print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.8.0+cu128
Device: cuda


In [2]:
# =============================================================================
# SEED
# =============================================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

## Dati

In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

df_sampled      = pd.read_parquet('../../preprocessing/electronics_review.parquet')
df_meta_aligned = pd.read_parquet('../../preprocessing/electronics_meta.parquet')

NUM_USERS    = df_sampled['user_id_int'].nunique()
NUM_ITEMS    = df_sampled['item_id_int'].nunique()
INTERACTIONS = len(df_sampled)
SPARSITY     = (1 - (INTERACTIONS / (NUM_USERS * NUM_ITEMS))) * 100

print(f"Utenti: {NUM_USERS}, Item: {NUM_ITEMS}, Interactions: {INTERACTIONS}")
print(f"Sparsity: {SPARSITY:.2f}%")

Utenti: 5439, Item: 27845, Interactions: 49202
Sparsity: 99.97%


In [ ]:
# =============================================================================
# CONFIG
# =============================================================================

ID_EMBED_DIM = 32    
HIDDEN_DIM   = 512   
OUTPUT_DIM   = 256   

print(f"NUM_ITEMS    = {NUM_ITEMS}")
print(f"ID_EMBED_DIM = {ID_EMBED_DIM}")
print(f"HIDDEN_DIM   = {HIDDEN_DIM}")
print(f"OUTPUT_DIM   = {OUTPUT_DIM}")
item_tower_params = (NUM_ITEMS * ID_EMBED_DIM
                     + ID_EMBED_DIM * HIDDEN_DIM + HIDDEN_DIM
                     + HIDDEN_DIM * OUTPUT_DIM + OUTPUT_DIM)
print(f"Item tower params: {item_tower_params:,}  "
      f"({item_tower_params * 4 / 1e6:.1f} MB)")

In [ ]:
class IDItemTower(nn.Module):
    def __init__(self, num_items, embed_dim=64, hidden_dim=512, output_dim=256):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, item_indices):  # [N] long → [N, output_dim]
        return F.normalize(self.net(self.item_emb(item_indices)), dim=-1)


class LocalScoreFunction(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, user_emb, item_emb):
        if user_emb.shape[0] != item_emb.shape[0]:
            if user_emb.shape[0] == 1:
                user_emb = user_emb.expand(item_emb.shape[0], -1)
            else:
                raise RuntimeError(f"Shape mismatch: {user_emb.shape} vs {item_emb.shape}")
        return self.net(torch.cat([user_emb, item_emb], dim=-1))


class ReLoGIDClean(nn.Module):
    def __init__(self, num_items,
                 embed_dim=64, hidden_dim=512, output_dim=256,
                 inference_temperature=0.07):
        super().__init__()
        self.item_tower = IDItemTower(num_items, embed_dim, hidden_dim, output_dim)
        self.client_mlp = LocalScoreFunction(output_dim * 2, hidden_dim=128)
        self.inference_temperature = inference_temperature

    def get_user_repr(self, train_item_ids_tensor):  # [N_pos] long
        item_embs = self.item_tower(train_item_ids_tensor)   # [N, output_dim]
        return item_embs.mean(dim=0, keepdim=True)           # [1, output_dim]

    def get_item_repr(self, item_indices):  # [N] long → [N, output_dim]
        return self.item_tower(item_indices)

    def training_score(self, user_repr, item_reprs):
        raw = self.client_mlp(user_repr, item_reprs).squeeze(-1)
        return raw / self.inference_temperature

    def score(self, user_repr, item_reprs):
        return self.training_score(user_repr, item_reprs)


def bpr_loss(pos_scores, neg_scores):
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)
    return -F.logsigmoid(diff).mean()


def make_model():
    return ReLoGIDClean(
        num_items=NUM_ITEMS,
        embed_dim=ID_EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        output_dim=OUTPUT_DIM,
        inference_temperature=0.07
    )


def is_shared(key):
    return key.startswith('item_tower.')


print("Architettura ReLoG-ID-Clean definita.")
tmp = make_model()
n_shared = sum(p.numel() for n, p in tmp.named_parameters() if is_shared(n))
n_local  = sum(p.numel() for n, p in tmp.named_parameters() if not is_shared(n))
print(f"  Shared parameters (item_tower): {n_shared:,}  ({n_shared*4/1e6:.2f} MB)")
print(f"  Local parameters   (client_mlp):  {n_local:,}  ({n_local*4/1e6:.2f} MB)")
print()
print("Comparison with original ReLoG:")
sbert_dim = 384
relog_shared = (sbert_dim*512+512 + 512*256+256)*2   # user+item tower
print(f"  ReLoG shared params (user+item tower): ~{relog_shared:,}")
print(f"  ReLoG-ID-Clean shared params (item_tower): {n_shared:,}")
del tmp

In [ ]:
def get_client_data(user_id, df, mode='train'):
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
    if len(user_df) < 3:
        return None, None
    indices        = user_df.index.tolist()
    train_idx      = indices[:-2]
    val_idx        = indices[-2]
    test_idx       = indices[-1]
    train_item_ids = user_df.loc[train_idx, 'item_id_int'].tolist()
    X_train = torch.tensor(train_item_ids, dtype=torch.long)
    if mode == 'val':
        target_id = int(user_df.loc[val_idx,  'item_id_int'])
    elif mode == 'test':
        target_id = int(user_df.loc[test_idx, 'item_id_int'])
    else:
        target_id = None
    return (X_train, train_item_ids), target_id


def get_fewshot_data(user_id, df, num_shots):
    user_df = df[df['user_id_int'] == user_id].sort_values('timestamp')
    if num_shots is None:
        if len(user_df) < 2:
            return None, None, None
        indices    = user_df.index.tolist()
        shot_idx   = indices[:-2]   
        target_idx = indices[-1]
    else:
        if len(user_df) < num_shots + 1:
            return None, None, None
        indices    = user_df.index.tolist()
        shot_idx   = indices[:num_shots]
        target_idx = indices[num_shots]
    shot_item_ids = user_df.loc[shot_idx, 'item_id_int'].tolist()
    X_shots       = torch.tensor(shot_item_ids, dtype=torch.long)
    target_id     = int(user_df.loc[target_idx, 'item_id_int'])
    return X_shots, shot_item_ids, target_id

In [46]:
# =============================================================================
# HARD NEGATIVE SAMPLING
# =============================================================================

def sample_hard_negatives(user_repr, pos_set, local_model,
                           num_neg, num_candidates, device, num_total_items):
    all_ids = np.arange(num_total_items)
    mask    = np.ones(num_total_items, dtype=bool)
    mask[np.array(list(pos_set), dtype=np.int64)] = False
    eligible = all_ids[mask]

    if len(eligible) < num_neg:
        return eligible.tolist()

    n_cands       = min(num_candidates, len(eligible))
    candidate_ids = np.random.choice(eligible, size=n_cands, replace=False)

    with torch.no_grad():
        cand_t      = torch.tensor(candidate_ids, dtype=torch.long, device=device)
        cand_reprs  = local_model.get_item_repr(cand_t)
        cand_scores = local_model.training_score(user_repr.detach(), cand_reprs)

    top_k       = min(num_neg, len(candidate_ids))
    top_indices = torch.topk(cand_scores, top_k).indices.cpu().numpy()
    return candidate_ids[top_indices].tolist()

## Training client

In [47]:
# =============================================================================
# LR SCHEDULE
# =============================================================================

def get_lr(base_lr, current_step, warmup_steps, total_steps):
    """LR warmup lineare → cosine annealing. Identico a ReLoG originale."""
    if current_step < warmup_steps:
        return base_lr * (current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

In [ ]:
# =============================================================================
# TRAIN CLIENT
# =============================================================================

def train_client(user_id, global_state_dict, X_train_items, train_item_ids,
                 device, client_states,
                 lr=0.0005, epochs=3, num_neg=10, use_hard_negatives=True,
                 lr_warmup_steps=10, current_step=0, total_steps=100):
    local_model = make_model().to(device)
    local_model.load_state_dict(global_state_dict, strict=True)

    # Ripristina MLP locale (se esiste)
    if client_states.get(user_id) is not None:
        local_model.client_mlp.load_state_dict(client_states[user_id])

    effective_lr = get_lr(lr, current_step, lr_warmup_steps, total_steps)
    optimizer    = torch.optim.Adam(local_model.parameters(), lr=effective_lr)
    local_model.train()

    X_train    = X_train_items.to(device)   # [N_pos] long
    pos_set    = set(train_item_ids)
    pos_tensor = torch.tensor(train_item_ids, dtype=torch.long, device=device)

    loss = None
    for _ in range(epochs):
        optimizer.zero_grad()

        pos_reprs = local_model.get_item_repr(pos_tensor)  # [N_pos, output_dim]
        N_pos     = pos_reprs.shape[0]

        user_repr = pos_reprs.mean(dim=0, keepdim=True)    # [1, output_dim]

        if use_hard_negatives:
            neg_ids = sample_hard_negatives(
                user_repr, pos_set, local_model,
                num_neg=len(train_item_ids) * num_neg,
                num_candidates=2000,
                device=device, num_total_items=NUM_ITEMS
            )
        else:
            all_ids = np.arange(NUM_ITEMS)
            mask    = np.ones(NUM_ITEMS, dtype=bool)
            mask[np.array(list(pos_set))] = False
            neg_ids = np.random.choice(all_ids[mask],
                                       size=len(train_item_ids) * num_neg,
                                       replace=True).tolist()

        neg_tensor = torch.tensor(neg_ids, dtype=torch.long, device=device)
        neg_reprs  = local_model.get_item_repr(neg_tensor)

      
        if N_pos > 1:
            sum_emb       = pos_reprs.sum(dim=0, keepdim=True)     # [1, D]
            user_repr_loo = (sum_emb - pos_reprs) / (N_pos - 1)    # [N_pos, D]
        else:
            user_repr_loo = user_repr.expand(N_pos, -1)

        pos_scores = local_model.training_score(user_repr_loo, pos_reprs)
        neg_scores = local_model.training_score(user_repr, neg_reprs)
        loss       = bpr_loss(pos_scores, neg_scores)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(local_model.parameters(), 1.0)
        optimizer.step()

    client_states[user_id] = local_model.client_mlp.state_dict()

    shared_state = {k: v.cpu() for k, v in local_model.state_dict().items()
                    if is_shared(k)}
    return shared_state, loss.item(), len(train_item_ids)

In [ ]:
def weighted_fedavg_momentum(global_model, local_weights_list, local_sizes,
                              momentum_buffer, beta=0.9):
    total_samples = sum(local_sizes)
    global_dict   = global_model.state_dict()
    keys_to_agg   = [k for k in global_dict if is_shared(k)]

    if momentum_buffer is None:
        momentum_buffer = {k: torch.zeros_like(global_dict[k]) for k in keys_to_agg}

    with torch.no_grad():
        for key in keys_to_agg:
            layer_avg = torch.zeros_like(global_dict[key])
            for i, w in enumerate(local_weights_list):
                layer_avg.add_(w[key].to(layer_avg.device),
                               alpha=local_sizes[i] / total_samples)
            delta = layer_avg - global_dict[key]
            momentum_buffer[key].mul_(beta).add_(delta, alpha=1 - beta)
            global_dict[key].add_(momentum_buffer[key])

    global_model.load_state_dict(global_dict, strict=True)
    return momentum_buffer

## Validation

In [ ]:
def evaluate_top_k(global_model, eval_users, df,
                   client_states, k=20, device='cuda', mode='test',
                   eval_fraction=1.0):
    k = int(k)
    if eval_fraction < 1.0:
        n_sample   = max(1, int(len(eval_users) * eval_fraction))
        eval_users = np.random.choice(eval_users, n_sample, replace=False)

    global_state = global_model.state_dict()
    all_ids_set  = set(range(NUM_ITEMS))
    all_ids_arr  = np.arange(NUM_ITEMS)
    use_amp      = (device == 'cuda')

    global_model.eval()
    all_item_idx = torch.arange(NUM_ITEMS, dtype=torch.long, device=device)
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_item_idx[i:i+chunk])
            for i in range(0, NUM_ITEMS, chunk)
        ], dim=0)   # [NUM_ITEMS, output_dim]

    hits, ndcgs, count = 0, 0, 0

    for user_id in tqdm(eval_users, desc=f'Evaluating ({mode})', leave=False):
        train_data, target_id = get_client_data(user_id, df, mode=mode)
        if train_data is None:
            continue
        X_train, train_ids = train_data   # X_train: long [N_pos]

        local_model = make_model().to(device)
        local_model.load_state_dict(global_state, strict=True)
        if client_states.get(user_id) is not None:
            local_model.client_mlp.load_state_dict(client_states[user_id])

        X_train_dev   = X_train.to(device)
        train_ids_set = set(train_ids)
        eligible_neg  = all_ids_arr[~np.isin(all_ids_arr, list(train_ids_set))]

        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=0.005)
        batch_pos = train_ids if len(train_ids) < 32 else random.sample(train_ids, 32)
        pos_t     = torch.tensor(batch_pos, dtype=torch.long, device=device)

        with torch.amp.autocast(device_type=device, enabled=use_amp):
            for _ in range(3):
                optimizer.zero_grad()
                user_repr = local_model.get_user_repr(X_train_dev)
                pos_embs  = all_item_embs[pos_t]
                n_neg     = min(len(batch_pos), len(eligible_neg))
                neg_idx   = np.random.choice(eligible_neg, size=n_neg, replace=False)
                neg_embs  = all_item_embs[torch.tensor(neg_idx, device=device)]
                loss = bpr_loss(
                    local_model.training_score(user_repr, pos_embs),
                    local_model.training_score(user_repr, neg_embs)
                )
                loss.backward()
                optimizer.step()

        local_model.eval()
        with torch.no_grad():
            user_repr  = local_model.get_user_repr(X_train_dev)
            neg_cands  = list(all_ids_set - train_ids_set - {target_id})
            neg_embs   = all_item_embs[np.array(neg_cands)]
            target_emb = all_item_embs[target_id].unsqueeze(0)

            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)

            all_scores = torch.cat([
                torch.tensor([target_score], device=device), neg_scores
            ])
            top_k_idx = torch.topk(all_scores, k).indices.cpu().numpy()
            if 0 in top_k_idx:
                hits += 1
                rank  = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1

        del local_model
        if device == 'cuda':
            torch.cuda.empty_cache()

    if count == 0:
        return 0.0, 0.0
    return hits / count, ndcgs / count

In [ ]:
def evaluate_fewshot(global_model, unseen_users, df,
                     num_shots, k=20, device='cuda',
                     finetune_epochs=5, lr=0.005):
    label = f'{num_shots}-shot' if num_shots is not None else 'full'
    k     = int(k)

    global_state = global_model.state_dict()
    all_ids_set  = set(range(NUM_ITEMS))
    all_ids_arr  = np.arange(NUM_ITEMS)
    use_amp      = (device == 'cuda')

    global_model.eval()
    all_item_idx = torch.arange(NUM_ITEMS, dtype=torch.long, device=device)
    with torch.no_grad():
        chunk = 4096
        all_item_embs = torch.cat([
            global_model.get_item_repr(all_item_idx[i:i+chunk])
            for i in range(0, NUM_ITEMS, chunk)
        ], dim=0)

    hits, ndcgs, count = 0, 0, 0

    for user_id in tqdm(unseen_users, desc=f'Few-shot eval ({label})', leave=False):
        X_shots, shot_item_ids, target_id = get_fewshot_data(user_id, df, num_shots)
        if X_shots is None:
            continue

        local_model = make_model().to(device)
        local_model.load_state_dict(global_state, strict=True)

        X_shots_dev  = X_shots.to(device)
        shot_ids_set = set(shot_item_ids)
        eligible_neg = all_ids_arr[~np.isin(all_ids_arr, list(shot_ids_set))]

        local_model.train()
        optimizer = torch.optim.Adam(local_model.client_mlp.parameters(), lr=lr)

        if len(shot_item_ids) > 0 and len(eligible_neg) > 0:
            batch_pos = (shot_item_ids if len(shot_item_ids) < 32
                         else random.sample(shot_item_ids, 32))
            pos_t = torch.tensor(batch_pos, dtype=torch.long, device=device)

            with torch.amp.autocast(device_type=device, enabled=use_amp):
                for _ in range(finetune_epochs):
                    optimizer.zero_grad()
                    user_repr = local_model.get_user_repr(X_shots_dev)
                    pos_embs  = all_item_embs[pos_t]
                    n_neg     = min(len(batch_pos), len(eligible_neg))
                    neg_idx   = np.random.choice(eligible_neg, size=n_neg, replace=False)
                    neg_embs  = all_item_embs[torch.tensor(neg_idx, device=device)]
                    loss = bpr_loss(
                        local_model.training_score(user_repr, pos_embs),
                        local_model.training_score(user_repr, neg_embs)
                    )
                    loss.backward()
                    optimizer.step()

        local_model.eval()
        with torch.no_grad():
            user_repr  = local_model.get_user_repr(X_shots_dev)
            neg_cands  = list(all_ids_set - shot_ids_set - {target_id})
            neg_embs   = all_item_embs[np.array(neg_cands)]
            target_emb = all_item_embs[target_id].unsqueeze(0)

            with torch.amp.autocast(device_type=device, enabled=use_amp):
                target_score = local_model.score(user_repr, target_emb).item()
                neg_scores   = local_model.score(user_repr, neg_embs)

            all_scores = torch.cat([
                torch.tensor([target_score], device=device), neg_scores
            ])
            top_k_idx = torch.topk(all_scores, k).indices.cpu().numpy()
            if 0 in top_k_idx:
                hits += 1
                rank  = int(np.where(top_k_idx == 0)[0][0])
                ndcgs += 1.0 / np.log2(rank + 2)
            count += 1

        del local_model
        if device == 'cuda':
            torch.cuda.empty_cache()

    if count == 0:
        return 0.0, 0.0
    print(f'  [{label}] utenti valutati: {count}/{len(unseen_users)}')
    return hits / count, ndcgs / count

## Split users

In [ ]:
def split_users(df, unseen_ratio=0.2, seed=42):
    rng = np.random.RandomState(seed)
    all_users = df['user_id_int'].unique()
    rng.shuffle(all_users)
    n_unseen     = int(len(all_users) * unseen_ratio)
    unseen_users = all_users[:n_unseen]
    train_users  = all_users[n_unseen:]
    return train_users, unseen_users

## Run experiment

In [ ]:
def run_experiment(seed):
    print(f'\n===== ReLoG-ID-Clean — seed {seed} =====')
    set_seed(seed)

    train_users, unseen_users = split_users(df_sampled, unseen_ratio=0.2, seed=seed)
    print(f'Train users: {len(train_users)},  Unseen: {len(unseen_users)}')

    LR                = 0.001
    LOCAL_EPOCHS      = 8 # 3
    NUM_NEG_TRAIN     = 10
    USE_HARD_NEG      = True
    CLIENTS_PER_ROUND = round(0.05 * len(train_users))
    GLOBAL_ROUNDS     = 100
    EVAL_EVERY        = 5
    FEDAVG_MOMENTUM   = 0.0
    K                 = 20
    EVAL_FRACTION     = 0.3
    LR_WARMUP_STEPS   = 10
    EARLY_STOP_PATIENCE = 10   # in termini di checkpoint, non round
    EARLY_STOP_MIN_DELTA = 1e-4
    early_stop_counter = 0

    client_states      = {uid: None for uid in train_users}
    best_val_hr        = 0.0
    best_val_ndcg      = 0.0
    best_state         = None
    best_client_states = None
    momentum_buffer    = None
    global_model       = make_model().to(device)

    print(f"\n=== Federated Training (ReLoG-ID-Clean) seed={seed} ===")
    print(f"Clients/round: {CLIENTS_PER_ROUND}  |  Rounds: {GLOBAL_ROUNDS}")
    print(f"{'Round':<6} | {'Loss':<8} | {'HR@'+str(K):<10} | {'NDCG@'+str(K):<10}")
    print('-' * 46)

    for round_num in range(1, GLOBAL_ROUNDS + 1):
        local_weights = []
        local_sizes   = []
        local_losses  = []

        round_state_dict = global_model.state_dict()
        selected = np.random.choice(train_users, CLIENTS_PER_ROUND, replace=False)

        for user_id in selected:
            train_data, _ = get_client_data(user_id, df_sampled)
            if train_data is None:
                continue
            X_train, train_item_ids = train_data

            w, loss, n = train_client(
                user_id, round_state_dict, X_train, train_item_ids,
                device, client_states,
                lr=LR, epochs=LOCAL_EPOCHS, num_neg=NUM_NEG_TRAIN,
                use_hard_negatives=USE_HARD_NEG,
                lr_warmup_steps=LR_WARMUP_STEPS,
                current_step=round_num - 1, total_steps=GLOBAL_ROUNDS
            )
            local_weights.append(w)
            local_sizes.append(n)
            local_losses.append(loss)

        if not local_weights:
            continue

        momentum_buffer = weighted_fedavg_momentum(
            global_model, local_weights, local_sizes,
            momentum_buffer, beta=FEDAVG_MOMENTUM
        )
        del local_weights

        if device == 'cuda' and round_num % 5 == 0:
            torch.cuda.empty_cache()

        avg_loss = sum(local_losses) / len(local_losses)

        if round_num % EVAL_EVERY == 0:
            val_hr, val_ndcg = evaluate_top_k(
                global_model, train_users, df_sampled,
                client_states, k=K, device=device, mode='val',
                eval_fraction=EVAL_FRACTION
            )
            marker = ''
            if val_hr > best_val_hr + EARLY_STOP_MIN_DELTA:
                best_val_hr        = val_hr
                best_val_ndcg      = val_ndcg
                best_state         = copy.deepcopy(global_model.state_dict())
                best_client_states = copy.deepcopy(client_states)
                early_stop_counter = 0
                marker = '  <- Best'
            else:
                early_stop_counter += 1
                if early_stop_counter >= EARLY_STOP_PATIENCE:
                    print(f'Early stop at round {round_num} (no improvement for {EARLY_STOP_PATIENCE} checkpoints)')
                    break
            print(f'{round_num:<6} | {avg_loss:<8.4f} | {val_hr:<10.4f} | {val_ndcg:<10.4f}{marker}')
        else:
            print(f'{round_num:<6} | {avg_loss:<8.4f} |')

    print('\n=== End Training ===')

    if best_state is None:
        print('[WARN] Nessun best_state — uso modello finale.')
        best_state         = global_model.state_dict()
        best_client_states = client_states

    global_model.load_state_dict(best_state)

    # TEST warm users
    print(f'\n--- TEST WARM USERS ---')
    warm_hr, warm_ndcg = evaluate_top_k(
        global_model, train_users, df_sampled,
        best_client_states, k=K, device=device, mode='test'
    )
    print(f'Warm  HR@{K}: {warm_hr:.4f}  |  NDCG@{K}: {warm_ndcg:.4f}')

    # TEST few-shot unseen users
    print(f'\n--- TEST UNSEEN USERS (few-shot) ---')
    shot_configs    = [1, 2, 3, None]
    fewshot_results = {}
    for num_shots in shot_configs:
        label = f'{num_shots}-shot' if num_shots is not None else 'full'
        hr, ndcg = evaluate_fewshot(
            global_model, unseen_users, df_sampled,
            num_shots=num_shots, k=K, device=device,
            finetune_epochs=5, lr=0.005
        )
        fewshot_results[label] = (hr, ndcg)
        print(f'  {label:<8}  HR@{K}: {hr:.4f}  |  NDCG@{K}: {ndcg:.4f}')

    return warm_hr, warm_ndcg, fewshot_results

## Main — multi-seed

In [ ]:
K           = 20
seeds       = [0]   # [0, 1, 2, 3, 4] 
shot_labels = ['1-shot', '2-shot', '3-shot', 'full']

warm_hrs,  warm_ndcgs  = [], []
fewshot_hrs   = {l: [] for l in shot_labels}
fewshot_ndcgs = {l: [] for l in shot_labels}

for s in seeds:
    warm_hr, warm_ndcg, fewshot_results = run_experiment(s)
    warm_hrs.append(warm_hr)
    warm_ndcgs.append(warm_ndcg)
    for label in shot_labels:
        fewshot_hrs[label].append(fewshot_results[label][0])
        fewshot_ndcgs[label].append(fewshot_results[label][1])

print('\n' + '=' * 62)
print('RESULTS ReLoG-ID-Clean (mesn ± std)')
print('=' * 62)
print(f"{'Scenario':<12} | {'HR@'+str(K):<24} | {'NDCG@'+str(K):<24}")
print('-' * 66)

m_hr   = np.mean(warm_hrs);    s_hr   = np.std(warm_hrs)
m_ndcg = np.mean(warm_ndcgs);  s_ndcg = np.std(warm_ndcgs)
print(f"{'warm':<12} | {m_hr:.4f} ± {s_hr:.4f}           | {m_ndcg:.4f} ± {s_ndcg:.4f}")

for label in shot_labels:
    m_hr   = np.mean(fewshot_hrs[label]);   s_hr   = np.std(fewshot_hrs[label])
    m_ndcg = np.mean(fewshot_ndcgs[label]); s_ndcg = np.std(fewshot_ndcgs[label])
    print(f"{label:<12} | {m_hr:.4f} ± {s_hr:.4f}           | {m_ndcg:.4f} ± {s_ndcg:.4f}")